# 06 · Decoder & submission

**Tujuan.** Menggabungkan keputusan bola hidup/mati (`03`) dan bukti tim (`05`) menjadi satu label per frame,
menulis file submission, memvalidasinya dengan validator resmi, dan mengukurnya dengan **proxy tanpa GT**.

| | |
|---|---|
| **Input** | `cache/dead.npz`, `cache/restarts.csv`, `cache/possession_model.npz`, `cache/tags.csv` |
| **Output** | `submissions/<tim>_<versi>.csv`, laporan `cache/reports/06_decoder_submission.*` |
| **Waktu** | ± 1 menit |

### Riwayat versi

| Versi | `D` | Bukti H/A | Catatan |
|---|---|---|---|
| v0 | `03` v3 | tim terdekat ke bola (±1 per frame), `SWITCH_HA` = 25 | acuan Fase 1 (`docs/experiment_log.md` #1) |
| v1 | `03` v3 | log-odds `05`: bentuk tim (out-of-fold) + β × tim terdekat | Fase 2.6 (#2) |
| v2 | `03` v4 (jangkar + aktivitas operan) | sama dengan v1 | Fase 3.1 (#3) |
| **v3** | **`03` v5** (+ syarat isyarat henti) | v1 + **prior restart**: 2 s pertama setelah corner/goal kick/free kick = tim pengambil | Fase 3.1b/3.2 — notebook ini |

### Decoder v1

Viterbi 3 keadaan (0 = `H`, 1 = `A`, 2 = `D`) per babak:

| Keadaan | Emisi di frame hidup | Emisi di frame `D` (`03`) |
|---|---|---|
| `H` | +½ · clip(`L_total`, ±`L_CLIP`) | −10⁴ |
| `A` | −½ · clip(`L_total`, ±`L_CLIP`) | −10⁴ |
| `D` | −10⁴ | +10⁴ |

Selisih emisi H − A per frame = log-odds. Mengganti H↔A membayar **`SWITCH_HA` = `SWITCH_S` × 25 × median |L|**:
bukti untuk tim baru harus setara ± `SWITCH_S` detik bukti "biasa". Biaya dinyatakan dalam **detik bukti**, sehingga
tidak bergantung pada skala log-odds model — dan tidak disetel di GT publik. Masuk/keluar `D` gratis karena `D` sudah
diputuskan `03`.

`clip` mencegah satu frame dengan log-odds ekstrem (mis. tracking rusak) mendominasi.

In [ ]:
%run ./common.ipynb

## Parameter

| Parameter | Nilai | Alasan |
|---|---|---|
| `VERSION` | `v3` | dicatat di nama file dan log eksperimen |
| `SWITCH_S` | 1,0 s | pergantian possession butuh kontrol yang bertahan (CHALLENGE.md); 1 s = skala `CTRL_MIN` ×2. Tidak disetel di GT publik — lihat tabel sensitivitas di bagian 4 |
| `L_CLIP` | 6 | batas log-odds per frame (P ≈ 0,998) |
| `RESTART_S` | 2,0 s | lama prior restart; 1 s terbukti memunculkan possession < 2 s buatan (`07` eksperimen D) |
| `BOOTSTRAP_N` | 1000 | interval bootstrap resmi untuk skor publik |

In [ ]:
VERSION = "v3"
RESTART_S = 2.0
SWITCH_S = 1.0
L_CLIP = 6.0
BOOTSTRAP_N = 1000
SUBMISSION_NAME = "{}_{}".format(TEAM_FILE, VERSION)

In [ ]:
rep = Report("06_decoder_submission")
with np.load(CACHE / "dead.npz") as dz:
    d_mask = dz["d_mask"]
with np.load(CACHE / "possession_model.npz") as mz:
    L_total, pseudo = mz["L_total"].astype(np.float64), mz["pseudo"]
restarts = pd.read_csv(CACHE / "restarts.csv", keep_default_na=False)
tags = load_tags()
live = GRADED & ~d_mask
L = np.clip(L_total, -L_CLIP, L_CLIP)
L_MED = float(np.median(np.abs(L[live])))
rep("versi {} | porsi D {:.3f} | median |L| di frame hidup {:.3f}".format(VERSION, d_mask[GRADED].mean(), L_MED))

## 1. Decoder

In [ ]:
SWITCH_COST = SWITCH_S * FPS * L_MED
L, n_restart = restart_prior(L, restarts, RESTART_S)        # common.ipynb §9
rep("prior restart: {} restart dengan tim pengambil diketahui, {} s".format(n_restart, RESTART_S))
with timer("viterbi"):
    labels = decode_possession(L, d_mask, SWITCH_COST)        # common.ipynb §9
rep("SWITCH_HA = {:.1f} (= {} s × {} × {:.3f})".format(SWITCH_COST, SWITCH_S, FPS, L_MED))

## 2. Proxy tanpa GT

Dihitung di seluruh match, tanpa label apa pun dari GT:

| Proxy | Arti | Catatan |
|---|---|---|
| `possession`, `durasi rata-rata` | jumlah & durasi possession yang dilaporkan | kewajaran; GT publik: 17 possession, 14,2 s (5 menit pertama saja) |
| `< 2 s %` | porsi possession sangat pendek | terlalu banyak = berkedip; nol = tak menangkap possession pendek (GT punya beberapa) |
| `pseudo-label cocok %` | label decoder = pseudo-label "kontrol jelas" (`05`) | sebagian melingkar (pemain terdekat ikut dalam bukti) — batas atas |
| `restart cocok %` | label 0,5 s setelah setiap `D` = tim restart terdekat (`03`) | konsistensi dengan temuan `02` #9 |
| `didukung sentuhan %` | porsi pergantian H↔A dengan sentuhan tim baru dalam ± 1 s | pergantian tanpa sentuhan mencurigakan |
| `porsi H hidup` | porsi H di waktu hidup | kewajaran (Barcelona di kandang) |

In [ ]:
with np.load(CACHE / "features_ha.npz") as fz:
    d_H, d_A = fz["d_H"], fz["d_A"]
observed = load_prep()["ball_src"] == 1


def proxies(lab):
    return possession_proxies(lab, live, pseudo, restarts, d_H, d_A, observed)      # common.ipynb §9


prox = proxies(labels)
rep("proxy {}: {}".format(VERSION, prox))

## 3. Tulis & validasi

In [ ]:
path, digest = write_submission(labels, SUBMISSION_NAME)
ok, summary = validate_file(path)
rep("file: {} | sha256 {}".format(path.relative_to(ROOT), digest))
rep("validasi resmi: {} | {}".format("LULUS" if ok else "GAGAL", summary))
assert ok, summary

## 4. Sensitivitas terhadap `SWITCH_S`

Untuk memilih **bukan** dengan GT publik, tabel ini memperlihatkan bagaimana proxy berubah. Kolom publik
disertakan sebagai informasi (tanpa bootstrap, jadi bacalah sebagai kasar; satu possession ≈ ±6 poin).
Nilai yang baik berada di **dataran** — proxy tidak berubah tajam di sekitarnya.

In [ ]:
sens = []
with timer("sensitivitas"):
    for s in (0.25, 0.5, 1.0, 1.5, 2.0, 3.0):
        lab_s = decode_possession(L, d_mask, s * FPS * L_MED)
        sc = score_labels_public(lab_s)
        sens.append({"SWITCH_S": s, **proxies(lab_s), "publik spellF1": round(sc["spell_f1_mean"], 3),
                     "publik macroF1": round(sc["macro_f1"], 3)})
sens = pd.DataFrame(sens).set_index("SWITCH_S")
rep.table(sens, "sensitivitas SWITCH_S:")

## 5. Cek kebenaran pada GT publik (dengan interval bootstrap)

In [ ]:
cols = ["spell_f1_mean", "spell_bootstrap_low", "spell_bootstrap_high", "macro_f1", "n_pred_spells",
        "n_gt_spells", "mean_spell_duration_s", "alive_dead_f1", "team_acc_gt_alive"]
board = {}
with timer("skor publik + bootstrap"):
    board[SUBMISSION_NAME] = score_public(path, bootstrap_n=BOOTSTRAP_N)
    for old in ("v2", "v1", "v0"):
        prev = SUBMISSIONS / "{}_{}.csv".format(TEAM_FILE, old)
        if prev.is_file():
            board[prev.stem] = score_public(prev, bootstrap_n=BOOTSTRAP_N)
    board["baseline_smooth50 (resmi)"] = score_public(
        OFFICIAL / "evaluation" / "results" / "baseline_smooth50.csv", bootstrap_n=BOOTSTRAP_N)
board_df = pd.DataFrame(board).T[cols].astype(float).round(4)
rep.table(board_df, "GT publik:")
rep.save({"version": VERSION, "file": str(path.relative_to(ROOT)), "sha256": digest, "summary": summary,
          "proxies": prox, "sensitivity": sens.reset_index().to_dict("records"),
          "public": board_df.reset_index().to_dict("records"),
          "params": {"SWITCH_S": SWITCH_S, "L_CLIP": L_CLIP, "L_MED": L_MED}})

## Ringkasan & keputusan (v3)

| GT publik | **v3** | v2 | v1 | v0 | baseline resmi |
|---|---|---|---|---|---|
| spellF1 [bootstrap 5–95%] | **± 0,64 [0,40 – 0,81]** | ± 0,59 [0,37 – 0,77] | ± 0,54 [0,32 – 0,74] | ± 0,34 [0,19 – 0,50] | ± 0,33 [0,21 – 0,45] |
| macroF1 | **± 0,87** | ± 0,86 | ± 0,77 | ± 0,72 | ± 0,62 |
| F1 hidup/mati | ± 0,96 | ± 0,95 | ± 0,88 | ± 0,88 | ± 0,90 |
| akurasi tim pada frame hidup GT | ± 0,90 | ± 0,89 | ± 0,75 | ± 0,69 | ± 0,83 |
| possession (GT 17) | 10 | 11 | 12 | 19 | 26 |

Seluruh match (proxy tanpa GT): ± 193 possession, rata-rata ± 17 s; pergantian didukung sentuhan ± 84%.

**Bacaan**

1. **Kenaikan v2 → v3 berasal dari `D` v5** (`D` palsu yang memecah possession panjang dibuang). Interval publik
   masih lebar; arah perubahan konsisten di semua versi.
2. **Prior restart** (2 s pertama setelah sepak pojok/tendangan gawang/tendangan bebas = tim pengambil) **netral** di
   jendela publik (dua restart bertag di sana sudah benar) dan menaikkan sedikit `didukung sentuhan` (`07` eksperimen D).
   Dasarnya terutama **definisi** (CHALLENGE.md: setiap restart punya penguasa yang jelas) dan arti tag yang
   diverifikasi geometri, bukan bukti empiris yang kuat — dicatat sebagai keputusan berbasis prinsip.
   Proxy `restart cocok %` (berbasis tim terdekat di akhir `D`) turun, tetapi proxy itu terbukti tidak andal untuk
   sepak pojok (`03`).
3. **Pembatas sekarang:** possession pendek yang tak terlihat bukti (`07` bagian 2) dan possession yang digabung
   karena `D` pendek terlewat.

**Keputusan:** v3 menjadi acuan (`docs/experiment_log.md` #7–#8).